## Imports

In [1]:
import pandas as pd
pd.options.display.float_format = '{:,.2f}'.format

In [ ]:
# import zipfile
# with zipfile.ZipFile('../data/raw2/Archive.zip', 'r') as zip_ref:
#     zip_ref.extractall('../data/raw2/')

## Load the dataset

In [4]:
df_orders = pd.read_csv('../data/raw/orders.csv')

In [5]:
# Reading data
df_orders_products_train = pd.read_csv('../data/raw/order_products_train.csv')

In [6]:
df_products = pd.read_csv("../data/raw/products.csv")

In [7]:
df_departments = pd.read_csv('../data/processed/departments_t.csv')

In [8]:
df_aisles = pd.read_csv('../data/raw/aisles.csv')

## Exploring the data

### orders

In [9]:
print(df_orders.columns)
print(df_orders.shape)
df_orders.head()


Index(['order_id', 'user_id', 'eval_set', 'order_number', 'order_dow',
       'order_hour_of_day', 'days_since_prior_order'],
      dtype='str')
(3421083, 7)


,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.00
2,473747,1,prior,3,3,12,21.00
3,2254736,1,prior,4,4,7,29.00
4,431534,1,prior,5,4,15,28.00


In [15]:
df_orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 3421083 entries, 0 to 3421082
Data columns (total 7 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   order_id                int64  
 1   user_id                 int64  
 2   eval_set                str    
 3   order_number            int64  
 4   order_dow               int64  
 5   order_hour_of_day       int64  
 6   days_since_prior_order  float64
dtypes: float64(1), int64(5), str(1)
memory usage: 182.7 MB


In [20]:
df_orders[['order_number', 'order_dow','order_hour_of_day', 'days_since_prior_order']].describe()

,order_number,order_dow,order_hour_of_day,days_since_prior_order
count,"3,421,083.00","3,421,083.00","3,421,083.00","3,214,874.00"
mean,17.15,2.78,13.45,11.11
std,17.73,2.05,4.23,9.21
min,1.00,0.00,0.00,0.00
25%,5.00,1.00,10.00,4.00
50%,11.00,3.00,13.00,7.00
75%,23.00,5.00,16.00,15.00
max,100.00,6.00,23.00,30.00


In [21]:
# missing values
df_orders.isna().sum()

order_id                       0
user_id                        0
eval_set                       0
order_number                   0
order_dow                      0
order_hour_of_day              0
days_since_prior_order    206209
dtype: int64

#### Analytical questions for`orders`

In [ ]:
# Exploring the eval_set distribution
df_orders['eval_set'].value_counts()

eval_set
prior    3214874
train     131209
test       75000
Name: count, dtype: int64

In [24]:
# How many unique orders exist?
df_orders["order_id"].nunique()

3421083

In [25]:
# How many distinct customers are represented in the dataset?
df_orders["user_id"].nunique()

206209

In [26]:
# Orders by day of week
df_orders["order_dow"].value_counts().sort_index()

order_dow
0    600905
1    587478
2    467260
3    436972
4    426339
5    453368
6    448761
Name: count, dtype: int64

In [27]:
# Orders by hour of day
df_orders["order_hour_of_day"].value_counts().sort_index()

order_hour_of_day
0      22758
1      12398
2       7539
3       5474
4       5527
5       9569
6      30529
7      91868
8     178201
9     257812
10    288418
11    284728
12    272841
13    277999
14    283042
15    283639
16    272553
17    228795
18    182912
19    140569
20    104292
21     78109
22     61468
23     40043
Name: count, dtype: int64

In [28]:
# Understanding missing values in days_since_prior_order
df_orders['days_since_prior_order'].isna().sum()

np.int64(206209)

In [ ]:
# Logical check: first orders and null values
df_orders.loc[df_orders["order_number"] == 1, "days_since_prior_order"].isna().all()

np.True_

In [30]:
# Is order_id unique?
df_orders["order_id"].is_unique

True

In [31]:
# Is order_dow in the correct range?
df_orders["order_dow"].between(0, 6).all()

np.True_

In [32]:
# Is order_hour_of_day in the correct range?
df_orders["order_hour_of_day"].between(0, 23).all()

np.True_

#### Filtering the `orders` table to train orders

In [33]:
df_orders_train = df_orders[df_orders["eval_set"] == "train"]
df_orders_train.head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
10,1187899,1,train,11,4,8,14.00
25,1492625,2,train,15,1,11,30.00
49,2196797,5,train,5,0,11,6.00
74,525192,7,train,21,2,11,6.00
78,880375,8,train,4,1,14,10.00


In [34]:
df_orders_train.drop(columns=["eval_set"], inplace=True)
df_orders_train.head()

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
10,1187899,1,11,4,8,14.00
25,1492625,2,15,1,11,30.00
49,2196797,5,5,0,11,6.00
74,525192,7,21,2,11,6.00
78,880375,8,4,1,14,10.00


In [35]:
## Delete df_orders to free up memory
del df_orders

### order_products_train

In [11]:
print(df_orders_products_train.columns)
print(df_orders_products_train.shape)
df_orders_products_train.head()

Index(['order_id', 'product_id', 'add_to_cart_order', 'reordered'], dtype='str')
(1384617, 4)


,order_id,product_id,add_to_cart_order,reordered
0,1,49302,1,1
1,1,11109,2,1
2,1,10246,3,0
3,1,49683,4,0
4,1,43633,5,1


### products

In [12]:
print(df_products.columns)
print(df_products.shape)
df_products.head()

Index(['product_id', 'product_name', 'aisle_id', 'department_id', 'prices'], dtype='str')
(49693, 5)


,product_id,product_name,aisle_id,department_id,prices
0,1,Chocolate Sandwich Cookies,61,19,5.80
1,2,All-Seasons Salt,104,13,9.30
2,3,Robust Golden Unsweetened Oolong Tea,94,7,4.50
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1,10.50
4,5,Green Chile Anytime Sauce,5,13,4.30


### departments_t

In [13]:
print(df_departments.columns)
print(df_departments.shape)
df_departments.head()

Index(['department_id', 'department'], dtype='str')
(21, 2)


,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol


### aisles

In [14]:
print(df_aisles.columns)
print(df_aisles.shape)
df_aisles.head()

Index(['aisle_id', 'aisle'], dtype='str')
(134, 2)


,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars
3,4,instant foods
4,5,marinades meat preparation
